# LoRA Fine-Tuning — end to end (Google Colab, free T4 GPU)

Fine-tune a small open-source model with **LoRA** (Hugging Face PEFT), then compare
**base vs fine-tuned** output and plot the training-loss curve.

**Bridge:** this is the open-source, hands-on version of Azure OpenAI fine-tuning — you
own the adapter, control the hyperparameters, and run it on a free GPU.

**Setup:** Runtime → Change runtime type → **T4 GPU**, then Run all.

> Uses TinyLlama (small, instruction-tuned) so it trains in minutes on free Colab.
> Swap `BASE_MODEL` for a larger model + QLoRA (4-bit) for a real task (see final cell).

## 1. Install dependencies

In [ ]:
!pip -q install "transformers>=4.44" "peft>=0.12" "datasets>=2.19" "accelerate>=0.33" bitsandbytes matplotlib

## 2. Load the base model + tokenizer from the Hugging Face Hub
The base weights will be **frozen** — LoRA only trains small adapters on top.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # small, fits free Colab

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token  # needed for batching

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
)
print(model.config.model_type, '-', sum(p.numel() for p in model.parameters())/1e9, 'B params')

## 3. Baseline: what does the model say BEFORE fine-tuning?
We'll teach it a specific persona/format, then compare against this.

In [ ]:
def generate(model, prompt, max_new_tokens=60):
    text = f"<|user|>\n{prompt}</s>\n<|assistant|>\n"
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

TEST_PROMPT = "What is the status of a late dealer invoice?"
print("BASE MODEL OUTPUT:\n", generate(model, TEST_PROMPT))

## 4. Apply LoRA with PEFT
Freeze the base, add tiny trainable adapters. Note the **trainable %** printed — ~0.1%.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                    # rank — adapter size/capacity
    lora_alpha=16,          # scaling (~ 2 x r)
    target_modules=["q_proj", "v_proj"],  # attention projections
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()   # e.g. 'trainable: 1.1M || all: 1.1B || 0.1%'

## 5. A tiny instruction dataset (behavior we want to teach)
Teach a concise, JMA-style dealer-support persona. Small set = shows the mechanics
(a real fine-tune uses 200-500 clean examples).

In [ ]:
from datasets import Dataset

examples = [
    {"q": "What is the status of a late dealer invoice?",
     "a": "A late invoice incurs a 2% penalty per month on the invoice total. Submit within 30 days of delivery."},
    {"q": "When is the dealer reserve released?",
     "a": "The dealer reserve is released once the retail contract meets its performance thresholds."},
    {"q": "What triggers a curtailment payment?",
     "a": "Vehicles unsold after 90 days trigger the first curtailment payment."},
    {"q": "How long to submit a warranty claim?",
     "a": "Warranty claims must be submitted through the dealer portal within 60 days of the repair."},
    {"q": "What prefix do Southeast dealer codes use?",
     "a": "Southeast dealer territory codes begin with the prefix ATL."},
    {"q": "What are standard parts payment terms?", "a": "Standard parts payment terms are Net 45."},
    {"q": "When do parts orders qualify for Net 60?",
     "a": "Orders over $50,000 qualify for Net 60 upon credit approval."},
]
# repeat a few times so the tiny set trains visibly (demo only)
examples = examples * 3

def to_text(ex):
    return {"text": f"<|user|>\n{ex['q']}</s>\n<|assistant|>\n{ex['a']}</s>"}

ds = Dataset.from_list([to_text(e) for e in examples])

def tokenize(batch):
    out = tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)
    out["labels"] = out["input_ids"].copy()
    return out

ds = ds.map(tokenize, batched=True, remove_columns=["text"])
print(len(ds), "training rows")

## 6. Train the adapters (HF Trainer)

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

args = TrainingArguments(
    output_dir="./lora-out",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    logging_steps=1,
    fp16=True,
    report_to="none",
)

collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)
trainer = Trainer(model=model, args=args, train_dataset=ds, data_collator=collator)
train_result = trainer.train()

## 7. Compare: base vs fine-tuned output (same prompt)

In [ ]:
print("FINE-TUNED OUTPUT:\n", generate(model, TEST_PROMPT))
print("\n(Compare to the BASE MODEL OUTPUT from step 3 — the persona/format should have shifted\n",
      "toward the concise, JMA-style answers we trained on.)")

## 8. Plot the training-loss curve
Should fall over steps. In a real run with a validation set, watch for validation loss
**rising** while training loss falls = overfitting.

In [ ]:
import matplotlib.pyplot as plt

losses = [log["loss"] for log in trainer.state.log_history if "loss" in log]
plt.figure(figsize=(6,3))
plt.plot(losses, marker=".")
plt.title("LoRA training loss")
plt.xlabel("logging step"); plt.ylabel("loss"); plt.grid(True, alpha=0.3)
plt.show()

## 9. Save the adapter (a few MB — NOT the whole model)

In [ ]:
model.save_pretrained("./jmf-lora-adapter")
print("Saved adapter. To reuse: load the base model, then")
print("  from peft import PeftModel; PeftModel.from_pretrained(base_model, './jmf-lora-adapter')")
!du -sh ./jmf-lora-adapter

## 10. For a real 7B model: QLoRA (4-bit) — reference
To fine-tune a larger model on the same free GPU, load the base in **4-bit** (QLoRA)
before applying LoRA:

```python
from transformers import BitsAndBytesConfig
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(BIG_MODEL, quantization_config=bnb, device_map="auto")
# then get_peft_model(model, lora_config) exactly as above
```

Same LoRA code, ~3x less memory — this is what puts a 7B fine-tune on a free Colab T4.

---
**Recap:** loaded a base model, froze it, trained ~0.1% of params as LoRA adapters on a
small dataset, saw the behavior shift, plotted the loss, and saved a few-MB adapter.
That's the open-source counterpart to Azure OpenAI fine-tuning — fine-tune for BEHAVIOR,
RAG for KNOWLEDGE.